In [39]:
from playwright.async_api import async_playwright, Error, TimeoutError
import pandas as pd
import time
import os
from pathlib import Path
import json


In [2]:
#Variables
host = "https://open.spotify.com/"
song_caracters = []
song_albunes = []
time_await_defaut = 1000
artist = 'Tyga'

In [ ]:
async with async_playwright() as p:

    try:
        print('Configuramos el navegador.')
        browser = await p.chromium.launch(headless=False)
        page = await browser.new_page()

        print(f'Redireccionamos al {host}')
        await page.goto(host, timeout=10000)
        await page.wait_for_timeout(time_await_defaut * 10)

    except TimeoutError:
        print(f'Timeout: El portal no cargo la url {host}')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

    print(f'Buscando a: {artist}')
    try:
        print(f'Ingresamos en el buscador: {artist}.')
        await page.locator('[role="combobox"]').click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.locator('[role="combobox"]').fill(f'{artist}')
        await page.wait_for_timeout(time_await_defaut * 10)

        #obtenemos la verificacion del artista a buscar de una manera mas desglozada
        print('Valdiando que exista el artista a buscar.')
        card_results = page.locator('[data-testid="top-result-card"]')
        a_tag = card_results.locator('a').first
        artist_tag = a_tag.locator('div')
        name_artist = await artist_tag.first.inner_text()
        await artist_tag.click()

    except TimeoutError:
        print('Timeout durante la buscar del artista.')

    except Error as e:
        print('Error Playwright durante la busqueda del artista:', e)

    except Exception as e:
        print('Error inesperado durante la bsuqueda del artista:', e)


    #Obtenemos la lista de las canciones mas reproducidas del artista y sus atributos.
    print('Obtenemos las lista de canciones mas escuchadas.')
    try:
        await page.get_by_text('Ver más').click()
        await page.wait_for_timeout(time_await_defaut * 3)
        card_list_songs = page.locator('[role="presentation"] div[data-testid="tracklist-row"]')
        print(await card_list_songs.count())

        for i in range(await card_list_songs.count()):

            atributos = card_list_songs.nth(i).locator('[role=gridcell] div[data-encore-id="text"]')
            
            if await atributos.count() == 3:
                #for atributo in range(await atributos.count()):
                song_name = await atributos.nth(0).inner_text()
                song_num_repoduccion = await atributos.nth(1).inner_text()
                song_time = await atributos.nth(2).inner_text()
            
                song_caracters.append({
                    'artist': artist,
                    'name song': song_name.replace(',','').strip(),
                    'num reproduction': song_num_repoduccion.replace(',','').strip(),
                    'time duration': song_time.replace(',','').strip()
                })

            else:
                raise('El numero de atributos cambio ya no es 3, revisar el portal')

    except TimeoutError:
        print('Timeout: no apareció el elemento')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

    #Obtenemos la discografia del artista y sus atributos.
    print('Obtenemos toda la discografia.')
    try:

        page_discografia = page.locator('[aria-label="Discografía"] span[data-encore-id="text"]')
        await page_discografia.click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.get_by_text("Fecha de lanzamiento").click()
        await page.wait_for_timeout(time_await_defaut * 3)
        await page.get_by_text("Cuadrícula").click()
        await page.wait_for_timeout(time_await_defaut * 5)

        all_discografia = page.locator('[role="presentation"] div[data-encore-id="card"]')
        for j in range(await all_discografia.count()):
            disc_name = await all_discografia.nth(j).locator('[data-encore-id="cardTitle"]').get_attribute('title')
            disc_type = await all_discografia.nth(j).locator('[data-encore-id="cardSubtitle"]').text_content()

            song_albunes.append({
                    'artist': artist,
                    'name album': disc_name.replace(',','').strip(),
                    'year': disc_type.split(' ')[0].replace(',','').strip(),
                    'type_disc': disc_type.split(' ')[-1].replace(',','').strip()
                })


    except TimeoutError:
        print('Timeout: no apareció el elemento')

    except Error as e:
        print('Error Playwright:', e)

    except Exception as e:
        print('Error inesperado:', e)

In [ ]:
df = pd.DataFrame(song_albunes)
df.to_csv(f'{artist}_albunes',encoding='utf8', index=True)


df = pd.DataFrame(song_caracters)
df.to_csv(f'{artist}_top_10',encoding='utf8', index=True)



In [ ]:
print('Generando archivo principal de artistas.')
path_fuente='/Users/axel/Documents/Portafolio'
archivos = [name for name in os.listdir(path_fuente) if name.endswith("top 10 of artist.csv")]
print(archivos)
for archivo in archivos:
    df_artist = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_artist = df_artist[df_artist["artist"] != "artist"]
    df_artist['num reproduction'] = df_artist['num reproduction'].astype(int)

df_grouped = df_artist.groupby("artist").agg({
    "num reproduction": "sum"
}).reset_index()

columas_new_add = [ 
            'real_name' ,
            'musical_genre',
            'artist_type' ,
            'start_carrer' ,
            'end_carrer' ,
            'number_albums' ,
            'followers' ,
            'monthly_listeners']

df_grouped = df_grouped.assign(**{col: '' for col in columas_new_add})
df_grouped.to_csv('list_artist.csv', encoding='utf-8', index=False, header=True)



In [ ]:
path_fuente='/Users/axel/Documents/Portafolio'
csv_artist = 'list_artist.csv'
csv_top_songs = 'top 10 of artist.csv'
csv_albunes = 'Albunes of artist.csv'


print(f'Leyendo archivo {csv_artist} para extraer index:')
archivos = [name for name in os.listdir(path_fuente) if name.endswith(csv_artist)]
for archivo in archivos:
    df_artist = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_artist = df_artist.where(pd.notnull(df_artist), None)

    artist_name = list(df_artist['artist'])
    artist_map = {artist: i for i, artist in enumerate(artist_name)}

print(f'Leyendo archivo {csv_top_songs} para informacion y añadir index:')
archivos = [name for name in os.listdir(path_fuente) if name.endswith(csv_top_songs)]
for archivo in archivos:
    df_top = pd.read_csv(f'{path_fuente}/{archivo}', encoding='utf-8', dtype=str)
    df_top = df_top.where(pd.notnull(df_top), None)
    
    df_top['id artist'] = df_top['artist'].map(artist_map)

data = list(df_top.itertuples(index=False, name=None))

data

In [ ]:
ruta = Path(os.getcwd())/'files_portafolio'
ruta.mkdir(exist_ok=True)
print(ruta.resolve())

In [13]:
def contar_palbras(path_fuente):

    with open(path_fuente, 'r' ) as archivo:
        texto = archivo.read()
        words = texto.replace('\n','').replace('\t','').replace('.',' ').split()
        print(len(words))

ruta = '/Users/axel/Documents/Portafolio/text_ejemplo.txt'
contar_palbras(ruta)

54


In [ ]:
## Lee la cintidad de palabras que esta dentro de texto ?

def min(text):
    
    new_cadena = text.lower().split()
    return new_cadena

def palabras_repetidas(cadena_low):
    diccionario = {}

    for palabra in (cadena_low):

        if palabra in diccionario:
            diccionario[palabra] +=1 
        else:
            diccionario[palabra] = 1       
    
    return diccionario


def main():
    cadena_low = min(text)
    diccionario = palabras_repetidas(cadena_low)
    print(diccionario)


if __name__ == "__main__":

    text = 'Gato sol mesa gato Nube mesa soL árbol gato nube mesa sol árbol gato'
    main()




{'gato': 4, 'sol': 3, 'mesa': 3, 'nube': 2, 'árbol': 2}


In [118]:
#haz una funcion que multiplique sin usar el simbolo de multiplicar

def milti_sin_simbolo(a,b):

    c = 0
    for i in range(b):
        c  += a

    return(c)

resultado = milti_sin_simbolo(3,9)
print(resultado)


27


In [154]:
# Realiza un programa:
# Que extraiga los dígitos de la siguiente cadena de texto.
# Que los ordene de menor a mayor y los devuelva en una cadena de Texto
# Que sume todos los digitos y devuelva el resultado de la suma total.
# Todos estos resultados deben ser mostrados por consola de manera simultánea

text = '3ha4sa2df3as5f3ad5a4sdf8df6'
numeros = ['0','1','2','3','4','5','6','7','8','9']
valores = []
suma = 0

for i in text:
    if i.isnumeric():
        
        valores.append(int(i))
        suma+=int(i)
valores.sort()
valores.reverse()
print(valores)
print(suma)


[8, 6, 5, 5, 4, 4, 3, 3, 3, 2]
43


In [185]:
# Crea una función que devuelva una lista ordenada de menor a mayor.
# No se pueden utilizar ni sorted() ni librerías.


array = [7,5,4,9,2,8,10,78,2,8,9]


def order_array(array):
    if len(array)<=1:
        return array

    pivote=array[0]
    menos = [x for x in array[1:] if x <= pivote]
    mayot = [x for x in array[1:] if x > pivote]

    return order_array(menos) + [pivote] + order_array(mayot)


print(order_array(array))

[2, 2, 4, 5, 7, 8, 8, 9, 9, 10, 78]


In [ ]:
#Sets, list, tuplas, diccionarios

list_frutas = ['manzana', 'sandia', 'uva']
tupla_verduras = ('papa', 'calabaza', 'zanahoria')
diccionario_comida = {'desayuno':'huevo_mexicana','comida':'milanesa_sopa', 'cena':'cafe_pan'}
set_cubiertos = {'cuchara', 'tenedor','cuchillo'}

print(type(list_frutas))
print(type(tupla_verduras))
print(type(diccionario_comida))
print(type(set_cubiertos))

print(tupla_verduras)
print(diccionario_comida)
print(set_cubiertos)



<class 'list'>
<class 'tuple'>
<class 'dict'>
<class 'set'>
('papa', 'calabaza', 'zanahoria')
{'desayuno': 'huevo_mexicana', 'comida': 'milanesa_sopa', 'cena': 'cafe_pan'}
{'tenedor', 'cuchillo', 'cuchara'}


In [34]:
#listas

#Convertir las demas estructuras a listas
print(list(tupla_verduras))
print(list(diccionario_comida)) #Para el diccionario devuelve solamente las claves
print(list(set_cubiertos))



['papa', 'calabaza', 'zanahoria']
['desayuno', 'comida', 'cena']
['tenedor', 'cuchillo', 'cuchara']


In [10]:
#tuplas

#Convertir las demas estructuras a tuplas (inmutables)
print(tuple(tupla_verduras))
print(tuple(diccionario_comida)) #Para el diccionario devuelve solamente las claves
print(tuple(set_cubiertos))

('papa', 'calabaza', 'zanahoria')
('desayuno', 'comida', 'cena')
('tenedor', 'cuchillo', 'cuchara')


In [ ]:
#sets

#Convertir las demas estructuras a sets
print(set(tupla_verduras))
print(set(diccionario_comida)) #Para el diccionario devuelve solamente las claves
print(set(set_cubiertos))

{'calabaza', 'papa', 'zanahoria'}
{'cena', 'comida', 'desayuno'}
{'tenedor', 'cuchillo', 'cuchara'}


In [ ]:
#diccionarios


#Concertir las demas estructuras a Diccionarios
diccionario_frutas = {}
diccionario_verduras = {}
diccionario_cubiertos = {}

#Recorremos mediante un for ya que el diccionario es calve:valor
for index, fruta in enumerate(list_frutas):
    diccionario_frutas[index] = fruta

for index, verdura in enumerate(tupla_verduras):
    diccionario_verduras[index] = verdura

for index, cubierto in enumerate(set_cubiertos):
    diccionario_cubiertos[index] = cubierto



print(diccionario_frutas)
print(diccionario_verduras)
print(diccionario_cubiertos)

{0: 'manzana', 1: 'sandia', 2: 'uva'}
{0: 'papa', 1: 'calabaza', 2: 'zanahoria'}
{0: 'tenedor', 1: 'cuchillo', 2: 'cuchara'}


In [42]:
#Con la estructura de datos que presetamos, crea un diccionario que tenga
#como claves: frutas, verduras, comidas, y cubiertos:

path_fuente='/Users/axel/Documents/Portafolio'

diccionario_global = {}
diccionario_general = {}

diccionario_global['frutas'] = list_frutas
diccionario_global['verduras'] = list(tupla_verduras)
diccionario_global['cubiertos'] = list(set_cubiertos)
diccionario_global['Comidas'] = diccionario_comida

diccionario_general['Axel_cocina'] = diccionario_global

print(json.dumps(diccionario_global))

with open(f'{path_fuente}/"cocina.json"', "w", encoding="utf-8") as f:
    json.dump(diccionario_general, f, ensure_ascii=False, indent=4)


{"frutas": ["manzana", "sandia", "uva"], "verduras": ["papa", "calabaza", "zanahoria"], "cubiertos": ["tenedor", "cuchillo", "cuchara"], "Comidas": {"desayuno": "huevo_mexicana", "comida": "milanesa_sopa", "cena": "cafe_pan"}}


In [59]:

def max_num(num1, num2):

    if num1 > num2:
        print(f'El numero mayor es: {num1}')

    elif num1 < num2:
        print(f'El numero mayor es: {num2}')

    else:  
        print('Error. Ambos numero son iguales')

num1 = 20
num2 = 10

print(f'El primer numero es: {num1}')
print(f'El segundo numero es: {num2}')

max(num1, num2)
 



El primer numero es: 20
El segundo numero es: 10


20

In [ ]:
def max_num(num1, num2):

    if num1 > num2:
        result = num1

    elif num1 < num2:
        result = num2

    else:  
        raise Exception ('Error. Ambos numero son iguales')
    
    return result

num1 = 10
num2 = 100

print(f'El primer numero es: {num1}')
print(f'El segundo numero es: {num2}')

resultado = max_num(num1,num2)
print(f'El numero mayor es: {resultado}')
 


El primer numero es: 10
El segundo numero es: 100
El numero mayor es: 100


In [ ]:
def max_num_tre(num1, num2, num3):

    def max_num(num1, num2):

        if num1 > num2:
            result = num1

        elif num1 < num2: 
            result = num2

        else:  
            raise Exception ('Error. Ambos numero son iguales')
        
        return result
  
    num_new = max_num(max_num(num1, num2), num3)

    return num_new

num1 = 10
num2 = 4
num3 = 6

resultado = max_num_tre(num1,num2, num3)

print(resultado)

10


In [98]:
def max_nums(array):
    
    def max_num(num1, num2):
        if num1 > num2:
            return num1

        elif num1 < num2:
            return num2
        else:  
           return num1


    if len(array) == 1:      
        return array[0]
    
    val1 = array[0]
    val2 = array[1]

    new_num = max_num(val1, val2)
    new_array = array[2:]
    new_array.append(new_num)

    return max_nums(new_array)


list_num = [10, 104, 1, 45, 500]
num_mayor = max_nums(list_num)
print(f'El numero mayor es: {num_mayor}')


El numero mayor es: 500


In [100]:
def max_num(num1, num2):
        if num1 > num2:
            return num1

        elif num1 < num2:
            return num2
        else:  
           return num1

def max_nums(array):
    if len(array) == 1:
        return array[0]

    return max_num(array[0], max_nums(array[1:]))

list_num = [800, 104, 1, 45, 500]
num_mayor = max_nums(list_num)
print(f'El numero mayor es: {num_mayor}')


El numero mayor es: 800


In [105]:
def max_nums(array,suma):
    if len(array) == 0:
        return suma

    suma = suma + array[0]

    return max_nums(array[1:],suma)

list_num = [800, 104, 1, 45, 500]
suma=0
num_mayor = max_nums(list_num,suma)
print(f'El ultimo valor del arreglo es : {num_mayor}')



El ultimo valor del arreglo es : 1450


In [108]:

import re
def vocal(caracter):
    vocales = ['u','o','i','e','a']

    if str(caracter) in vocales:
        return True
    else:
        return False        
    


print(vocal('8'))

False


In [132]:

def sum_list(array):
    result = array[0]
    for numero in array[1:]:
        result += numero
    return result

def mul_lista(array):
    result = array[0]
    for numero in array[1:]:
        result *= numero
    return result


lista = [147,798, 396, 132, 714]
print(f'La suma de la lista es: {sum_list(lista)}')
print(f'La multipliacacion de la lista es: {mul_lista(lista)}')

La suma de la lista es: 2187
La multipliacacion de la lista es: 4378118931648


In [135]:

def inversion(cadena):

    cadena_inversa = ''
    for caracter in reversed(cadena):
        cadena_inversa += caracter   

    return str(cadena_inversa)

cadena ='estoy probando'
print(inversion(cadena))


odnaborp yotse


In [148]:
def palindromo(cadena):

    cadena_inversa = ''
    for caracter in reversed(cadena):
        if caracter != ' ':
            cadena_inversa += caracter   

    if cadena.replace(' ','').lower() == cadena_inversa.lower():
        return True
    else:
        return False

print(palindromo('OSo'))

True
